# Weight interfaces

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TuLLI-AI/symprog/blob/main/notebooks/weight_interface.ipynb)

A `Solver` stores weights in a **weight manager**, which also implements the arithmetic that the semirings use during inference (`add`, `mult`, `log`, `max`, `neg`, ...). Swapping the weight manager changes how inference is computed, e.g., numerically with PyTorch or symbolically with SymPy, without changing anything else.

| Manager | Values | Gradients |
| --- | --- | --- |
| `pytorch` (default) | PyTorch tensors | autograd |
| `python` | Python/NumPy floats | no |
| `sympy`, `symengine` | symbolic expressions | exact symbolic derivatives |
| `jax` (**experimental**) | JAX arrays | `jax.grad`, of functions of the weights |

In a solver, the manager is chosen with `solver.add_weights(..., container_type=...)`. Below we use managers directly via `BuildWeights`.

## Setup

If `symprog` is not installed, the notebook uses the version in this repository.

In [1]:
# %pip install -q "symprog[notebooks]"

## use the installed symprog, or else the version in this repository
try:
    import symprog
except ImportError:
    import pathlib, sys
    repo = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "symprog" / "__init__.py").exists()), None)
    if repo is None:
        raise ImportError("symprog is not installed: pip install symprog, or run this notebook from the symprog repository")
    sys.path.insert(0, str(repo))
    import symprog

import sympy
from IPython.display import Math, display

from symprog.solvers import BuildWeights

weights = [0.5, 0.6]         # the weight values
var_map = {"a": 0, "b": 1}   # variable name -> position in `weights`

## PyTorch

Each manager stores the weights, and `assignment` returns the weight of a variable:

In [2]:
w = BuildWeights(manager="pytorch", weights=weights, wmap=var_map, requires_grad=True)
print(w)
w.assignment("a")

PytorchWeights(a=0.5, b=0.6)


tensor(0.5000, grad_fn=<SelectBackward0>)

Its operations are implemented with the underlying numerical library, so they are differentiable. Below we compute $f = \log w(a) \cdot (1 - w(b))$ (where `neg` is $1 - x$) and its gradient:

In [3]:
a, b = w.assignment("a"), w.assignment("b")
f = w.mult(w.log(a), w.neg(b))

print("f =", f)
w.gradient(f)

f = tensor(-0.2773, grad_fn=<MulBackward0>)


{'weights': tensor([0.8000, 0.6931])}

## Python

The `python` manager uses plain Python and NumPy numbers (without gradients):

In [4]:
p = BuildWeights(manager="python", weights=weights, wmap=var_map)
p.mult(p.log(p.assignment("a")), p.neg(p.assignment("b")))

np.float64(-0.2772588722239781)

## Symbolic weights

The `sympy` and `symengine` managers represent each weight by a symbol, so the same operations build symbolic expressions, with exact symbolic gradients:

In [5]:
sym = BuildWeights(manager="sympy", weights=weights, wmap=var_map, symbolic=True)
a, b = sym.assignment("a"), sym.assignment("b")
f = sym.mult(sym.log(a), sym.neg(b))

display(Math("f = " + sympy.latex(f)))
sym.gradient(f, key="weights")

<IPython.core.display.Math object>

Matrix([
[(1 - b)/a],
[  -log(a)]])

## JAX (experimental)

The `jax` manager uses [JAX](https://docs.jax.dev/en/latest/) arrays (requires `pip install "symprog[jax]"`). Since JAX differentiates *functions*, its `gradient` takes a function of the weight vector:

In [6]:
import jax.numpy as jnp

j = BuildWeights(manager="jax", weights=weights, wmap=var_map)
print("f =", j.mult(j.log(j.assignment("a")), j.neg(j.assignment("b"))))

j.gradient(lambda x: jnp.log(x[0]) * (1 - x[1]), key="weights")

f = -0.27725884


Array([0.79999995, 0.6931472 ], dtype=float32)

All four computations agree, and the numerical gradients match the symbolic one, $\big((1 - b)/a,\ -\log a\big) = (0.8, 0.693)$.

## How managers are used

During inference, a semiring calls these operations to combine the weights along the circuit (see *Implementing new algebras* in `solver_tutorial.ipynb`). For instance, the probability semiring uses `add` and `mult`, and the Viterbi semiring uses `max` and `mult`. In a solver, the manager is selected with `container_type`:

```python
solver.add_weights({"a": 0.5, "b": 0.6}, container_type="python")
```